In [1]:
from azure.cosmos import CosmosClient, exceptions, PartitionKey
import json
from azure.identity import AzureCliCredential
import os
import openai_helper 
import nest_asyncio
import asyncio
from dotenv import load_dotenv

load_dotenv()

# Define your Cosmos DB account information
endpoint = os.environ["AZURE_COSMOSDB_ENDPOINT"]


# Initialize the Cosmos client
client = CosmosClient(endpoint, credential=AzureCliCredential())
database = client.get_database_client(os.environ["AZURE_COSMOSDB_DBNAME"])
print("Connected to Cosmos DB at:", endpoint)
print("Database name:", database.id)
print("Container name:", os.environ["AZURE_COSMOSDB_CONTAINER_NAME"])

questions = [
"Give me a list of books published in the year 2000",
"Give me a list of book of travel category",
"Give me a list of books from author Agatha Christie",
"Give me some haunted incidents from california state",
"Give me some wines found in Italy",
"Give me wines tasted by Roger Voss",
"Give me some wines in the variety of Red Blend",
"Give me some business category news",
"Give me a list of students graduated in the year 2025"]

Connected to Cosmos DB at: https://anildwacosmoswestus.documents.azure.com:443/
Database name: booksdb
Container name: books_perftest


## Pure Full-Text Search

In [5]:
container_name = "books_perftest" 
embedding_model = "text-embedding-ada-002" 
question = questions[0]

print(question)
async def full_text_search(container_name, search_query, top_k=5):
    def run_query():
        try:
            container = database.get_container_client(container_name)
            search_query_arr = search_query.split(" ")
            #print(search_query_arr)
            query_string = f"""
            SELECT TOP @top_k c.fileName
            FROM c
            ORDER BY RANK FullTextScore(c.text, '{search_query}')
            """

            items = container.query_items(
                query=query_string,
                parameters=[
                    {"name": "@top_k", "value": top_k},
                ],
                enable_cross_partition_query=True
            )
            return [item for item in items]
        except Exception as e:
            print(f"Error in query: {e}")
            return []

    return await asyncio.to_thread(run_query)

items = await full_text_search(container_name, question, top_k=5)

for item in items:
    print(item)
    #print(item['fileName'])
    #query_with_filename(item['fileName'])
    #print(item['textSimilarityScore'])
    #print("***************")

Give me a list of books published in the year 2000
{'fileName': 'wine_de_Trafford_2015_Cape_Winemakers_Guild_Perspective_Merlot-Cabernet_Sauvignon_(Stellenbosch).csv'}
{'fileName': 'wine_de_Trafford_2008_Straw_Wine_Chenin_Blanc_(Stellenbosch).csv'}
{'fileName': 'wine_àMaurice_2012_William_Ivey_Red_(Columbia_Valley_(WA)).csv'}
{'fileName': "wine_Zusslin_NV_Brut_Zero_Sans_Soufre_Sparkling_(Crémant_d'Alsace).csv"}
{'fileName': "wine_Zusslin_2013_Brut_Zéro_Sparkling_(Crémant_d'Alsace).csv"}


## Pure Vector Search

In [7]:
container_name = "books_perftest" 
embedding_model = "text-embedding-ada-002" # text-embedding-ada-002
top_k = 5
threshold = 0.7
search_query = questions[3]
embedding_result = await openai_helper.get_embeddings([search_query], model=embedding_model)
search_query_embedded = embedding_result.data[0].embedding

print(search_query)
container = database.get_container_client(container_name)    
items = container.query_items( 
    query="""
    SELECT top @top_k c.fileName, VectorDistance(c.textVector, @embedding) AS textSimilarityScore 
    FROM c
    
    ORDER BY VectorDistance(c.textVector, @embedding) 
    """, 
    parameters=[
        {"name": "@embedding", "value": search_query_embedded},
        {"name": "@top_k", "value": top_k},
        {"name": "@threshold", "value": threshold}
    ], 
    enable_cross_partition_query=True)

for item in items:
    print(item)

Give me some haunted incidents from california state
{'fileName': "cultural_data_Les_Paris_d'Orsay.csv", 'textSimilarityScore': 0.09590354788230354}
{'fileName': 'news_Apple_iPod_family_expands_market.csv', 'textSimilarityScore': 0.08847368426294107}
{'fileName': "news_Apple_laptop_is_'greatest_gadget'.csv", 'textSimilarityScore': 0.08685281083195978}
{'fileName': 'cultural_data_Steam_Museum.csv', 'textSimilarityScore': 0.08641825923564739}
{'fileName': "news_Norway_upholds_'Napster'_ruling.csv", 'textSimilarityScore': 0.08442800407381901}


## RRF Query

In [6]:
container_name = "books_perftest" 
embedding_model = "text-embedding-ada-002" 
question = questions[0]

print(question)

async def search_with_rrf(container_name, embedding_model, search_query, top_k=5, threshold=0.7):
    embedding_result = await openai_helper.get_embeddings([search_query], model=embedding_model)
    search_query_embedded = embedding_result.data[0].embedding
  # already a list of floats
    keywords = ' '.join(f'"{word}"' for word in search_query.split())
    def run_query():
        try:
            container = database.get_container_client(container_name)
            search_query_arr = search_query.split(" ") #['published', 'in', 'the', 'year', '2000']
            items = container.query_items(
                query=f"""
                SELECT TOP {top_k} c.fileName, c.text
                FROM c
                ORDER BY RANK RRF(
                    FullTextScore(c.text, '{search_query}'),
                    VectorDistance(c.textVector, {search_query_embedded})
                )
                """,
                parameters=[],
                enable_cross_partition_query=True
            )
            return [item for item in items]
        except Exception as e:
            print(f"Error in query: {e}")
            return []

    return await asyncio.to_thread(run_query)

items = await search_with_rrf(container_name, embedding_model, question, top_k=5, threshold=0.7)


# VectorDistance(c.textVector, {search_query_embedded})
for item in items:
    print(item)
    #print(item['fileName'])
    #query_with_filename(item['fileName'])
    #print(item['textSimilarityScore'])
    #print("***************")

Give me a list of books published in the year 2000
{'fileName': 'news_Spears_seeks_aborted_tour_payment.csv', 'text': 'category: entertainment.\nfilename: 247.txt.\ntitle: Spears seeks aborted tour payment.\ncontent:  Singer Britney Spears is suing eight insurance companies that have refused to reimburse her for cancelling her 2004 world tour.  The pop star cancelled her Onyx Hotel tour last June after suffering a knee injury during a video shoot. She is now seeking to be reimbursed for the tour\'s £5m ($9.3m) costs in a claim filed at New York State Supreme Court. Seven London-based companies and an eighth Paris firm have been given up to 30 days to respond to the complaint.  The 22-year-old star initially missed a number of shows on the 82-date tour after injuring her knee during a show in Illinois last March. But she was rushed to hospital and needed surgery after a later incident while filming a video for her song Outrageous, leading her to cancel the rest of the tour, including da